# An implementation of Multi-Task Learning using convolutional neural networks.

In [17]:
import torch
import torch.nn as nn 
import torch.nn.functional as F 
from torch.utils.data import Dataset, DataLoader 
import dotenv as de

import numpy as np
import pandas as pd 
from sklearn.preprocessing import StandardScaler, LabelEncoder 
from sklearn.model_selection import train_test_split

In [18]:
class TicketDataset(Dataset):
    def __init__(self, df, numeric_cols, categorical_cols, reg_target, clf_target, scalar=None, encoders=None, fit=True):
        self.numeric_cols = numeric_cols
        self.categorical_cols = categorical_cols

        num_data = df[numeric_cols].values.astype(np.float32)
        if fit:
            self.scalar = StandardScaler()
            num_data = self.scalar.fit_transform(num_data)
        else:
            self.scalar = scalar
            num_data = self.scaler.transform(num_data)

        self.num_data = torch.tensor(num_data, dtype=torch.float32)


        self.cat_data = []
        self.encoders = encoders if encoders else {}
        cat_encoded = []
        for col in categorical_cols:
            if fit:
                enc = LabelEncoder()
                encded = enc.fit_transform(df[col].astype(str))
                self.encoders[col] = enc 
            else:
                enc = self.encoders[col]
                encoded = df[col].astype(str).map(
                    lambda x: enc.transform([x])[0] if x in enc.classes_ else 0
                ).values

            cat_encoded.append(encoded)

        self.cat_data = torch.tensor(
            np.stack(cat_encoded, axis=1), dtype=torch.long
        ) if categorical_cols else torch.empty((len(df), 0), dtype=torch.long)


        #Targets
        self.reg_y = torch.tensor(df[reg_target].values, dtype=torch.float32)
        self.clf_y = torch.tensor(df[clf_target].values, dtype=torch.long)

    def __len__(self):
        return len(self.reg_y)

    def __getitem__(self, index):
        return (self.num_data[index], self.cat_data[index], self.reg_y[index], self.clf_y[index])

In [19]:
class MultiTaskNet(nn.Module):
    def __init__(
            self,
            num_numeric,
            cat_cardinalities,
            n_classes=3,
            trunk_dims=(64,32),
            emb_dropout=0.1,
            trunk_dropout=0.2
    ):

        super.__init__()

        self.embeddings = nn.ModuleList([
            nn.Embedding(card, min(50, (card + 1) // 2))
            for card in cat_cardinalities 
        ])
        emb_total = sum(min(50, (c + 1) // 2) for c in cat_cardinalities)
        self.emb_dropout = nn.Dropout(emb_dropout)

        input_dim = num_numeric + emb_total 

        trunk_layers = []
        prev = input_dim
        for dim in trunk_dims:
            trunk_layers += [
                nn.Linear(prev, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU(),
                nn.Dropout(trunk_dropout)
            ]
            prev=dim 

        self.trunk = nn.Sequential(*trunk_layers)
        trunk_out = prev

        # --- Task Heads --- 
        self.reg_head = nn.Sequential(
            nn.Linear(trunk_out, trunk_out // 2), nn.ReLU(),
            nn.Linear(trunk_out // 2, 1) 
        )
        self.clf_head = nn.Sequential(
            nn.Linear(trunk_out, trunk_out // 2), nn.ReLU(),
            nn.Linear(trunk_out // 2, n_classes)
        )

    def forward(self, x_num, x_cat):
        if len(self.embeddings) > 0:
            embs = [emb(x_cat[:, 1]) for i, emb in enumerate(self.embeddings)]
            x_cat_emb = self.emb_dropout(torch.cat(embs, dim=1))
            x = torch.cat([x_num, x_cat_emb], dim = 1)
        else:
            x = x_num 

        shared = self.trunk(x)
        reg_out = self.reg_head(shared).squeeze(-1)


class UncertaintyWeightedLoss(nn.Module):
    def __init__(self):
        super().__init__()

        self.log_var_reg = nn.Parameter(torch.zeros(1))
        self.log_var_clf = nn.Parameter(torch.zeros(1))

    def forward(self, reg_pred, reg_true, clf_logits, clf_true):
        reg_loss = F.mse_loss(reg_pred, reg_true)
        clf_loss = F.cross_entropy(clf_logits, clf_true)

        loss = (
            torch.exp(-self.log_var_reg) * reg_loss + self.log_var_reg
            + torch.exp(-self.log_var_clf) * clf_loss + self.log_var_clf
        )
        return loss.squeeze(), reg_loss.item(), clf_loss.item()

In [20]:
def train(model, loss_fn, train_loader, val_loader, epochs=50, lr=1e-3):
    optimizer = torch.optim.Adam(
        list(model.parameters()) + list(loss_fn.parameters()), lr=lr
    )

    device = 'cuda' if torch.cuda.is_available() else 'cpu' 
    model.to(device); loss_fn.to(device)

    for epoch in range(epochs):
        model.train()
        for x_num, x_cat, reg_y, clf_y in train_loader:
            x_num, x_cat = x_num.to(device), x_cat.to(device)
            reg_y, clf_y = reg_y.to(device), clf_y.to(device)

            optimizer.zero_grad()
            reg_pred, clf_logits = model(x_num, x_cat)
            loss, r_loss, c_loss = loss_fn(reg_pred, reg_y, clf_logits, clf_y)
            loss.backward()
            optimizer.step()

        # Validation
        model.eval()
        val_mae, val_acc, n = 0, 0, 0
        with torch.no_grad():
            x_num, x_cat = x_num.to(device), x_cat.to(device)
            reg_y, clf_y = reg_y.to(device), clf_y.to(device)
            reg_pred, clf_logits = model(x_num, x_cat)
            val_mae += (reg_pred - reg_y).abs().sum().item()
            val_acc += (clf_logits.argmax(1) == clf_y).sum().item()
            n += len(reg_y)
        print(f"Epoch {epoch+1:3d} | reg_loss {r_loss:.3f} "
              f"clf_loss {c_loss:.3f} | val MAE {val_mae/n:.3f} "
              f"val Acc {val_acc/n:.3f}")

In [21]:
DATASET_NAME = de.get_key(".env", "DATASET_NAME")

data = pd.read_excel(f"data/{DATASET_NAME}", sheet_name = [1,2]) #Load in the 2nd and 3rd sheets of the excel file (all other sheets contain metadata and are not relevant)
DATA_ORIGINAL = pd.concat([data[1], data[2]], axis=0, ignore_index=True)
def preprocess_dataframe(df_orig):
    df = df_orig.copy() #Copy the original dataset

    df['Fan_Mailing_List'] = df['Fan_Mailing_List'].map({"Yes" : 1, "No" : 0}).astype(bool)
    df['Seat_Is_Upper'] = df['Seat_Location'].map({"Upper" : 1, "Lower" : 0}).astype(bool)

    df['Days_Before_Game'] = df['Days_Before_Game'].astype("float64")

    return df 

DATA = preprocess_dataframe(DATA_ORIGINAL)

In [25]:
numeric_cols = ['Num_Tickets_Purchased', 'Age', 'Ticket_Price']
categorical_cols = ['Seat_Location', 'Fan_Mailing_List']

train_df, temp_df = train_test_split(
    DATA, test_size=0.3, random_state=42,
    stratify=DATA['Customer_Type']
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42,
    stratify=temp_df['Customer_Type']
)

train_ds = TicketDataset(train_df, numeric_cols, categorical_cols, 'Days_Before_Game', 'Customer_Type', fit=True)
val_ds = TicketDataset(val_df, numeric_cols, categorical_cols, 'Days_Before_Game', 'Customer_Type', fit=False)
test_ds = TicketDataset(test_df, numeric_cols, categorical_cols, 'Days_Before_Game', 'Customer_Type', fit=False)

cat_cards = [train_df[c].nunique() for c in categorical_cols]

model = MultiTaskNet(num_numeric=len(numeric_cols),
                     cat_cardinalities=cat_cards, n_classes=3)

loss_fn = UncertaintyWeightedLoss()

train(model, loss_fn,
    DataLoader(train_ds, batch_size=64, shuffle=True),
    DataLoader(val_ds, batch_size=256)
)

UnboundLocalError: cannot access local variable 'encoded' where it is not associated with a value